Model Testing with Datasets 
------------


Imports
------------

In [ ]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from skimage.feature import hog, local_binary_pattern
from scipy.stats import kurtosis, skew
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


Paths & Directory Setup
------------


In [ ]:
# Paths
datasetDir_path = "../Dataset/Processing/AdvanceTestingDataset"

target_genuine = os.path.join(datasetDir_path, "TestingDataset/Genuine")
target_forged = os.path.join(datasetDir_path, "TestingDataset/Forged")

# Output directories for processed images
processed_genuine = os.path.join(datasetDir_path, "Pre-ProcessedDataset/Genuine")
processed_forged = os.path.join(datasetDir_path, "Pre-ProcessedDataset/Forged")

# Ensure output directories exist
os.makedirs(os.path.join(datasetDir_path, "Pre-ProcessedDataset"), exist_ok=True)
os.makedirs(processed_genuine, exist_ok=True)
os.makedirs(processed_forged, exist_ok=True)

print("Paths & Directory setup done")

Definitions
------------

In [ ]:
# Image processing function
def process_image(image_path, output_path, size=(256, 256)):
    try:
        # Read image
        image = cv2.imread(image_path)

        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Apply Gaussian Blur for noise reduction
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)

        # Resize to 256x256
        resized = cv2.resize(blurred, size, interpolation=cv2.INTER_AREA)

        # Save processed image
        cv2.imwrite(output_path, resized)
    except Exception as e:
        print(f"Error processing {image_path}: {e}")

In [ ]:
def preprocess_image(image_path, output_path, size=(256, 128)):
    """Preprocess an image: Convert to grayscale, denoise, resize, and apply binarization."""
    
    # Read image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Check if the image was loaded properly
    if img is None:
        print(f"Skipping {image_path}: Unable to read.")
        return

    # Apply Gaussian Blur for noise reduction
    img = cv2.GaussianBlur(img, (3, 3), 0)

    # Resize image
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)

    # Apply adaptive thresholding (better for signature images)
    img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                cv2.THRESH_BINARY_INV, 11, 2)

    # Save processed image
    cv2.imwrite(output_path, img)

In [ ]:
def process_dataset(source_folder, target_folder):
    """Processes all images in a folder and saves them to the target folder."""
    
    # Ensure target directory exists
    os.makedirs(target_folder, exist_ok=True)

    # Process images with tqdm progress bar
    for file_name in tqdm(os.listdir(source_folder), desc=f"Processing {source_folder}"):
        input_path = os.path.join(source_folder, file_name)
        output_path = os.path.join(target_folder, file_name)
        preprocess_image(input_path, output_path)

Definitions for Feature Extraction

In [ ]:
def extract_hog_features(img):
    """Extracts Histogram of Oriented Gradients (HOG) features."""
    features, _ = hog(img, pixels_per_cell=(16, 16), cells_per_block=(2, 2), 
                      orientations=9, visualize=True, block_norm='L2-Hys')
    return features

In [ ]:
def extract_lbp_features(img):
    """Extracts Local Binary Pattern (LBP) features."""
    lbp = local_binary_pattern(img, P=24, R=3, method="uniform")
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 27), range=(0, 26))
    hist = hist.astype("float")
    hist /= hist.sum()  # Normalize histogram
    return hist

In [ ]:
def extract_hu_moments(img):
    """Extracts shape-based Hu Moments features."""
    moments = cv2.moments(img)
    hu_moments = cv2.HuMoments(moments).flatten()
    hu_moments = -np.sign(hu_moments) * np.log10(np.abs(hu_moments))  # Log scale
    return hu_moments

In [ ]:
def extract_orb_features(img):
    """Extracts ORB keypoints and descriptors."""
    orb = cv2.ORB_create()
    keypoints, descriptors = orb.detectAndCompute(img, None)
    return len(keypoints)  # Number of keypoints detected

In [ ]:
def extract_pixel_intensity(img):
    """Extracts basic pixel intensity statistics."""
    return [np.mean(img), np.std(img), np.median(img), skew(img.flatten()), kurtosis(img.flatten())]

In [ ]:
def extract_pressure_features(img):
    """Estimates pressure points based on stroke thickness & pixel density."""
    binary = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    thickness = np.sum(binary == 255) / np.count_nonzero(binary)  # Ratio of ink pixels
    density = np.sum(binary == 255) / binary.size  # Fraction of inked area
    return [thickness, density]

In [ ]:
def extract_features(image_path):
    """Extracts all features from a given image."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img = cv2.resize(img, (256, 128))  # Resize to fixed size
    
    features = []
    features.extend(extract_hog_features(img))
    features.extend(extract_lbp_features(img))
    features.extend(extract_hu_moments(img))
    features.append(extract_orb_features(img))
    features.extend(extract_pixel_intensity(img))
    features.extend(extract_pressure_features(img))
    
    return features

In [ ]:
def process_dataset(input_folder, label):
    """Extracts features from all images in a dataset directory."""
    data = []
    for file_name in tqdm(os.listdir(input_folder), desc=f"Extracting Features from {label}"):
        image_path = os.path.join(input_folder, file_name)
        features = extract_features(image_path)
        if features:
            data.append([file_name, label] + features)
    return data

Model Testing

In [ ]:
# Load and preprocess images
def load_images(image_paths):
    images = []
    for path in image_paths:
        img = tf.keras.preprocessing.image.load_img(path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        img = tf.keras.applications.inception_v3.preprocess_input(img)
        images.append(img)
    return np.array(images)

Process
------------


In [ ]:
# Process genuine images
print("Processing Genuine Signatures...")
for file_name in tqdm(os.listdir(target_genuine)):
    input_path = os.path.join(target_genuine, file_name)
    output_path = os.path.join(processed_genuine, file_name)
    process_image(input_path, output_path)

# Process forged images
print("Processing Forged Signatures...")
for file_name in tqdm(os.listdir(target_forged)):
    input_path = os.path.join(target_forged, file_name)
    output_path = os.path.join(processed_forged, file_name)
    process_image(input_path, output_path)

print("Preprocessing Complete! Processed images are saved in 'ProcessedDataset'.")


Feature Extraction
------------


In [ ]:
# Extract features for both classes
genuine_data = process_dataset(processed_genuine, 0)  # Label 0 for genuine
forged_data = process_dataset(processed_forged, 1)    # Label 1 for forged

# Combine and save as CSV
columns = ["filename", "label"] + [f"feature_{i}" for i in range(len(genuine_data[0]) - 2)]
df = pd.DataFrame(genuine_data + forged_data, columns=columns)
df.to_csv("../Dataset/Features/testing_signature_features.csv", index=False)

print("Feature extraction completed and saved!")

Model Testing
---

In [ ]:
# Load handcrafted features
data = pd.read_csv("../Dataset/Features/testing_signature_features.csv")
labels = data["label"].values
handcrafted_features = data.iloc[:, 2:].values  # Skip filename and label columns

In [ ]:
# Load images for CNN
image_size = (299, 299)  # InceptionV3 default input size
image_paths = [f"/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Dataset/Processing/AdvanceTestingDataset/Pre-ProcessedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}" for file, label in zip(data["filename"], labels)]

In [ ]:
# Load images
images = load_images(image_paths)

model = load_model("/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Model/signature_forgery_detection_model.h5")


In [ ]:

# Predict on test set
y_pred_probs = model.predict([images, handcrafted_features.reshape(-1, handcrafted_features.shape[1], 1)])
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions

# Compute evaluation metrics
accuracy = accuracy_score(labels, y_pred)
precision = precision_score(labels, y_pred)
recall = recall_score(labels, y_pred)
f1 = f1_score(labels, y_pred)
conf_matrix = confusion_matrix(labels, y_pred)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)


If not upper one worked and gives errors, even it's not check this too

In [ ]:
# Ensure handcrafted_features is correctly reshaped
handcrafted_features = handcrafted_features.reshape(images.shape[0], handcrafted_features.shape[1], 1)

# Make predictions
y_pred_probs = model.predict([images, handcrafted_features])
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions

# Check prediction shape
print("y_pred_probs shape:", y_pred_probs.shape)  # Should be (batch_size, 1)

# Compute evaluation metrics using actual labels
accuracy = accuracy_score(labels, y_pred)
precision = precision_score(labels, y_pred)
recall = recall_score(labels, y_pred)
f1 = f1_score(labels, y_pred)
conf_matrix = confusion_matrix(labels, y_pred)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
